# Do-calculus on all collapsed graphs (from hcm_to_collapsed.ipynb)

Build HSCM -> collapse -> (optional) augment -> (optional) marginalize; run
identify_effect for each case and show a summary table. Software tests are in `tests/`.

In [1]:
import do_calculus as dc
from collapsed_cases import COLLAPSED_DO_CALCULUS_CASES, build_cgm_for_case

Confounder CGM nodes: ['U', 'Q^{y|a}', 'Q^a']
Confounder CGM edges: [('U', 'Q^a'), ('U', 'Q^{y|a}')]
Confounder Interferer CGM nodes: ['U', 'Z', 'Q^a', 'Q^{y|a}']
Confounder Interferer CGM edges: [('U', 'Q^{y|a}'), ('U', 'Q^a'), ('Z', 'Q^{y|a}'), ('Q^a', 'Z')]
Instrument CGM nodes: ['Q^z', 'U', 'Q^{a|z}', 'Y']
Instrument CGM edges: [('Q^z', 'Y'), ('U', 'Q^{a|z}'), ('U', 'Y'), ('Q^{a|z}', 'Y')]
['U', 'Q^{y|a}', 'Q^a', 'Q^y']
[('U', 'Q^a'), ('U', 'Q^{y|a}'), ('Q^{y|a}', 'Q^y'), ('Q^a', 'Q^y')]
Confounder: P(Q^y | do(Q^a)) identified as:
\sum_{Qy_a}{P\left(Qy_a\right) \cdot P\left(Qy\mid Qa,Qy_a\right)}
['U', 'Z', 'Q^a', 'Q^{y|a}', 'Q^y']
[('U', 'Q^{y|a}'), ('U', 'Q^a'), ('Z', 'Q^{y|a}'), ('Q^a', 'Q^y'), ('Q^{y|a}', 'Q^y'), ('Q^y', 'Z')]
['Q^z', 'U', 'Q^{a|z}', 'Y', 'Q^y', 'Q^{y|a}', 'Q^a']
[('Q^z', 'Y'), ('U', 'Q^{a|z}'), ('U', 'Y'), ('Q^{a|z}', 'Y'), ('Q^{y|a}', 'Q^y'), ('Q^a', 'Q^y')]
Instrument CGM nodes: ['Q^z', 'U', 'Q^{a|z}', 'Y', 'Q^a']
Instrument CGM edges: [('Q^z', 'Q^a'), ('U',

In [2]:
if not dc.PYAGNUM_AVAILABLE:
    print("pyagrum not installed.")
else:
    results = []
    for case in COLLAPSED_DO_CALCULUS_CASES:
        name = case[0]
        expected_id = case[10]
        cgm, unobserved, Y, X, _ = build_cgm_for_case(dc, case)
        res = dc.identify_effect(cgm, Y=Y, X=X, unobserved=unobserved)
        ok = res.identifiable == expected_id
        results.append((name, expected_id, res.identifiable, ok, res.formula_latex or res.error or ""))
    for name, exp, got, ok, detail in results:
        status = "ok" if ok else "MISMATCH"
        short = (detail[:60] + "...") if detail and len(detail) > 60 else (detail or "")
        print("{}: expected_id={} got={} {} | {}".format(name, exp, got, status, short))

confounder_aug: expected_id=True got=True ok | \sum_{Qy_a}{P\left(Qy_a\right) \cdot P\left(Qy\mid Qa,Qy_a\r...
confounder_interferer_aug: expected_id=True got=False MISMATCH | [pyAgrum] Directed cycle detected: Add a directed cycle in a...
instrument_mar: expected_id=True got=True ok | \sum_{Qa_z}{P\left(Y\mid Qa,Qa_z\right) \cdot P\left(Qa_z\ri...
ID_ex3_collapse: expected_id=True got=True ok | \sum_{Qw_a}{P\left(Y\mid Qa,Qw_a\right) \cdot P\left(Qw_a\ri...
ID_ex2_aug: expected_id=True got=True ok | \sum_{Qy_a_z,Qz_a}{P\left(Qy_a_z,Qz_a\right) \cdot P\left(Qy...
ID_ex6_collapse: expected_id=True got=True ok | \sum_{Qz_a,W}{P\left(Y\mid Qa,Qz_a,W\right) \cdot P\left(W\m...
ID_ex4_aug: expected_id=True got=True ok | \sum_{Qa_z}{P\left(Qa_z\right) \cdot P\left(W\mid Qa,Qa_z\ri...
ID_ex1_mar: expected_id=True got=True ok | \sum_{Qa,Qw_a_z}{P\left(Y\mid Qa,Qw,Qw_a_z\right) \cdot P\le...
ID_ex5_mar: expected_id=True got=True ok | \sum_{Qa_x_z}{P\left(Y\mid Qa_x,Qa_x_z\right) \cdot P\left(Q.